migrate Project 3 logic to Databricks serverless

In [0]:
from pyspark.sql import functions as F

LANDING_PATH = "/Volumes/nyc_taxi/landing/raw_files/yellow_tripdata/"
ZONE_PATH    = "/Volumes/nyc_taxi/landing/raw_files/reference/taxi_zone_lookup.csv"

In [0]:
bronze = (
    spark.read.parquet(LANDING_PATH)
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("source_month", F.regexp_extract("source_file", r"(\d{4}-\d{2})", 1))
    # 阶段 1：为了和 Project 3 对账，暂时沿用"从文件名取月份"的做法，阶段 2 修正
    .withColumn("pickup_year",  F.split("source_month", "-").getItem(0).cast("int"))
    .withColumn("pickup_month", F.split("source_month", "-").getItem(1).cast("int"))
)

In [0]:
# 维度表1:zone lookup(265行)
zone_lookup = (
    spark.read.csv(ZONE_PATH,
                   header=True, inferSchema=True)
    .select("LocationID", "Borough", "Zone", "service_zone")
)

# PULocationID和DOLocationID都指向同一张zone表,所以要分别重命名、join两次
pu_zone = zone_lookup.select(
    F.col("LocationID").alias("PULocationID"),
    F.col("Borough").alias("pickup_borough"),
    F.col("Zone").alias("pickup_zone"),
)
do_zone = zone_lookup.select(
    F.col("LocationID").alias("DOLocationID"),
    F.col("Borough").alias("dropoff_borough"),
    F.col("Zone").alias("dropoff_zone"),
)

# 维度表2:ratecode lookup(官方只有6种,手写一个小表,不用额外下载文件)
RATECODE_ROWS = [
    (1, "Standard rate"),
    (2, "JFK"),
    (3, "Newark"),
    (4, "Nassau or Westchester"),
    (5, "Negotiated fare"),
    (6, "Group ride"),
]
ratecode_lookup = spark.createDataFrame(
    RATECODE_ROWS, ["RatecodeID", "rate_code_desc"])

In [0]:
enriched = (
    bronze
    .join(F.broadcast(pu_zone), on="PULocationID", how="left")
    .join(F.broadcast(do_zone), on="DOLocationID", how="left")
    .join(F.broadcast(ratecode_lookup), on="RatecodeID", how="left")
)

display(enriched.select("PULocationID", "pickup_borough", "pickup_zone",
    "DOLocationID", "dropoff_borough", "dropoff_zone",
    "RatecodeID", "rate_code_desc",).limit(5))

In [0]:
cleaned = enriched.filter(
    (F.col("trip_distance") > 0)
    & (F.col("fare_amount") >= 0)
    & (F.col("total_amount") > 0)
    & (F.col("passenger_count") > 0)
    & F.col("tpep_pickup_datetime").isNotNull()
    & F.col("tpep_dropoff_datetime").isNotNull()
    & (F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
)

In [0]:
summary = (
    cleaned.groupBy("pickup_year", "pickup_month", "pickup_borough")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
    )
    .orderBy("pickup_year", "pickup_month", F.desc("trip_count"))
)

display(summary)

In [0]:
bronze_count = bronze.count()
silver_count = cleaned.count()
dropped = bronze_count - silver_count

print(f"bronze rows : {bronze_count:,}   (Project 3: 22,519,712)")
print(f"silver rows : {silver_count:,}   (Project 3: 21,964,002)")
print(f"dropped     : {dropped:,}   ratio {dropped / bronze_count:.2%}   (Project 3: 2.47%)")